# Project 4 — Time-Series Forecasting

**Goal:** Forecast a synthetic time series with lag features and Gradient Boosting, using a chronological split.

Run the cells from top to bottom. Data is generated locally or loaded from scikit-learn; no CSV download is required.

## Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, silhouette_score, mean_absolute_error, mean_squared_error,
    precision_score, recall_score
)
from sklearn.datasets import load_breast_cancer, make_classification, make_blobs
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
print("Setup complete")


## Project 4 — Time-Series Forecasting (Lag Features + Gradient Boosting)
**Goal:** Predict a synthetic series using past observations. The split is chronological to avoid training on future observations.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n = 500
t = np.arange(n)
series = 0.03 * t + 3 * np.sin(2 * np.pi * t / 30) + rng.normal(0, 0.8, n)
ts = pd.DataFrame({"time": t, "value": series})

plt.figure(figsize=(10, 4))
plt.plot(ts["time"], ts["value"])
plt.title("Synthetic time series")
plt.xlabel("Time")
plt.ylabel("Value")
plt.show()

In [ ]:
# Build supervised learning rows from lagged values
for lag in [1, 2, 3, 7, 14]:
    ts[f"lag_{lag}"] = ts["value"].shift(lag)
ts["rolling_mean_7"] = ts["value"].shift(1).rolling(7).mean()
model_df = ts.dropna().reset_index(drop=True)

feature_cols = [c for c in model_df.columns if c.startswith("lag_")] + ["rolling_mean_7"]
split = int(len(model_df) * 0.8)
train_df, test_df = model_df.iloc[:split], model_df.iloc[split:]
X_train, y_train = train_df[feature_cols], train_df["value"]
X_test, y_test = test_df[feature_cols], test_df["value"]

ts_model = GradientBoostingRegressor(random_state=RANDOM_STATE)
ts_model.fit(X_train, y_train)
forecast = ts_model.predict(X_test)

mae = mean_absolute_error(y_test, forecast)
rmse = np.sqrt(mean_squared_error(y_test, forecast))
print("MAE:", round(mae, 3))
print("RMSE:", round(rmse, 3))

plt.figure(figsize=(10, 4))
plt.plot(test_df["time"], y_test, label="Actual")
plt.plot(test_df["time"], forecast, label="Predicted")
plt.xlabel("Time")
plt.ylabel("Value")
plt.title("Time-series forecasting: actual vs predicted")
plt.legend()
plt.show()

## Possible extensions
- Compare with a naive forecast.
- Evaluate with rolling time windows.